- 001 hello world
- 002 GPU encoding & similarity calc
- 003 LLM determination
- 004 max values only; redo LLM calculation using vLLM for speed/parallelization. nevermind, the LLM optimization did not work. 

In [ ]:
from ppp_paths import databulk, dataint
usellm = False

In [ ]:
import pandas as pd
import unicodedata
import re
import gc

In [ ]:
possiblepairs = pd.read_parquet(dataint + 'potentialpairswithinwindow.parquet')

In [ ]:
import unicodedata
import re

def normalize(data):
    normal = unicodedata.normalize('NFKD', data).encode('ASCII', 'ignore')
    val = normal.decode("utf-8")
    # Lower case
    val = val.lower()
    # remove special characters
    val = re.sub('[^A-Za-z ]+', ' ', val)
    # remove multiple spaces
    val = re.sub(' +', ' ', val)
    # remove trailing spaces
    val = val.strip()
    return val


In [ ]:
possiblepairs_papersonly = possiblepairs['magid'].drop_duplicates().reset_index(drop=True)
possiblepairs_papersonly
institutions = pd.read_csv(databulk + "paperauthoridaffiliationname.zip", sep='\t', compression='zip', dtype=str)
institutions['authorid'] = institutions['authorid'].astype(str)
institutions = institutions.rename(columns={'paperid': 'magid'})
institutions['magid'] = institutions['magid'].astype(str)
institutions = institutions[['magid', 'affiliationame']]
# authors = pd.read_csv(databulk + "paperauthororder.zip", sep='\t', compression='zip', dtype=str)
# authors = authors.rename(columns={'paperid': 'magid'})
# authors['magid'] = authors['magid'].astype(str)
# authors['authorid'] = authors['authorid'].astype(str)
# authors = authors[['magid', 'authorid']]
# authors.dropna(inplace=True)
institutions = pd.merge(institutions, possiblepairs_papersonly, on='magid', how='inner')
institutions = institutions.dropna()
institutions = institutions.drop_duplicates()
del possiblepairs_papersonly 
gc.collect()
# authors
# authors.dropna()
# authors = pd.merge(authors, institutions, on=['magid', 'authorid'], how='left')
# del institutions 
# gc.collect()
# authors = authors[['magid', 'affiliationame']]


In [ ]:
institutions['affiliationame'] = institutions['affiliationame'].astype(str)
institutions['affiliationame'] = institutions['affiliationame'].apply(normalize)
institutions = institutions.drop_duplicates()
institutions = institutions.dropna()
institutions = institutions[institutions['affiliationame'] != 'nan']
institutions


In [ ]:
paperinstitutions = institutions
print(paperinstitutions['affiliationame'].value_counts())
paperinstitutions = paperinstitutions[paperinstitutions['affiliationame'] != 'nan']
print(paperinstitutions['affiliationame'].value_counts())


In [ ]:
possiblepairs_patentsonly = possiblepairs['patent_id'].drop_duplicates().reset_index(drop=True)
possiblepairs_patentsonly
assignees = pd.read_csv(databulk + 'g_assignee_disambiguated.tsv.zip', sep='\t', compression='zip')
assignees = pd.merge(assignees, possiblepairs_patentsonly, on='patent_id', how='inner')
assignees = assignees[['patent_id', 'disambig_assignee_organization']]
assignees = assignees.dropna()
assignees = assignees.drop_duplicates()
assignees['disambig_assignee_organization'] = assignees['disambig_assignee_organization'].astype(str)
assignees['disambig_assignee_organization'] = assignees['disambig_assignee_organization'].apply(normalize)
assignees

In [ ]:
del possiblepairs_patentsonly
gc.collect()

In [ ]:
possiblepairsassigneesinstitutions = possiblepairs.copy()
possiblepairsassigneesinstitutions = pd.merge(possiblepairsassigneesinstitutions, assignees, on='patent_id', how='left')
possiblepairsassigneesinstitutions = pd.merge(possiblepairsassigneesinstitutions, paperinstitutions, on='magid', how='left')
del assignees 
del paperinstitutions
gc.collect()
# possiblepairsassigneesinstitutions = possiblepairsassigneesinstitutions.dropna()
possiblepairsassigneesinstitutions = possiblepairsassigneesinstitutions.drop_duplicates()
possiblepairsassigneesinstitutions

In [ ]:
possiblepairsassigneesinstitutions.to_parquet(dataint + 'TEMPpossiblepairsassigneesinstitutions.parquet', compression='snappy')

(if only doing the LLM,need to run the above cells to set up possiblepairsassigneesinstitutions)

In [ ]:
# possiblepairsassigneesinstitutions['affiliationame'] = possiblepairsassigneesinstitutions['affiliationame'].apply(normalize)
# possiblepairsassigneesinstitutions['disambig_assignee_organization'] = possiblepairsassigneesinstitutions['disambig_assignee_organization'].apply(normalize)

In [ ]:
assigneeinstitution = possiblepairsassigneesinstitutions[['disambig_assignee_organization', 'affiliationame']]
assigneeinstitution

In [ ]:
assigneeinstitution = assigneeinstitution.drop_duplicates()
assigneeinstitution = assigneeinstitution.dropna()
assigneeinstitution['id'] = assigneeinstitution.index
assigneeinstitution['id'] =assigneeinstitution['id'].astype(str)
assigneeinstitution.to_parquet(dataint+'assigneeinstitution.parquet')

this is a hack to avoid rewriting the gpu embedding code

In [ ]:
assigneeinstitutionexport = assigneeinstitution.copy()
assigneeinstitutionexport = assigneeinstitutionexport.rename(columns={'affiliationame':'title', 'disambig_assignee_organization':'abstract'})
assigneeinstitutionexport = assigneeinstitutionexport[['id', 'title', 'abstract']]
assigneeinstitutionexport.to_csv(dataint + 'institutionstoembed.tsv', sep='\t', index=None)
 


In [ ]:
import pandas as pd
import numpy as np
import torch
from sentence_transformers import SentenceTransformer
import logging
import time
from tqdm import tqdm
import os
import multiprocessing as mp
from functools import partial
import gc 

# Configure logging
logging.basicConfig(format='%(asctime)s - %(message)s', level=logging.INFO)

input_file = dataint + "institutionstoembed.tsv"  # Replace with your input file
output_file = dataint + "embeddedinstitutions.parquet"  # Replace with your output file
# input_file = "pppapptoembed.tsv"  # Replace with your input file
# output_file = "encoded_patents.parquet"

chunk_size=1000000
encode_batch_size=4096
# encode_batch_size=8192
# encode_batch_size=16384
# encode_batch_size=32768
# encode_batch_size=65536
encode_chunk_size=200000
num_workers=12



In [ ]:
def encode_chunk(df_chunk, model, batch_size=encode_batch_size, chunk_size=encode_chunk_size):
    """
    Encode abstracts and titles for a dataframe chunk.
    """
    # Filter non-null abstracts
    df_chunk = df_chunk[df_chunk["abstract"].notnull()][["id", "title", "abstract"]]
    
    if df_chunk.empty:
        return None

    # Encode abstracts
    logging.info(f"Doing abstracts for {len(df_chunk)} rows")
    df_chunk["abstract"] = df_chunk["abstract"].fillna("").astype(str)
    abstracts = df_chunk["abstract"].tolist()
    with torch.cuda.amp.autocast():  # Mixed precision
        emb_abstracts = model.encode(
            abstracts,
            batch_size=batch_size,
            show_progress_bar=True,
            convert_to_numpy=True,
            device="cuda",
            normalize_embeddings=True,
            chunk_size=chunk_size
        )
    df_chunk["encoded_abstract"] = emb_abstracts.tolist()  # Direct assignment

    # Encode titles
    logging.info(f"Doing titles for {len(df_chunk)} rows")
        # Ensure titles are strings and not NaN
    df_chunk["title"] = df_chunk["title"].fillna("").astype(str)
    titles = df_chunk["title"].tolist()
    with torch.cuda.amp.autocast():
        emb_titles = model.encode(
            titles,
            batch_size=batch_size,
            show_progress_bar=True,
            convert_to_numpy=True,
            device="cuda",
            normalize_embeddings=True,
            chunk_size=chunk_size
        )
    df_chunk["encoded_title"] = emb_titles.tolist()

    # Round embeddings to 3 decimals
    df_chunk["encoded_abstract"] = df_chunk["encoded_abstract"].apply(lambda x: np.around(x, decimals=3))
    df_chunk["encoded_title"] = df_chunk["encoded_title"].apply(lambda x: np.around(x, decimals=3))

    return df_chunk[["id", "encoded_title", "encoded_abstract"]]


In [ ]:
def worker(chunk, model, batch_size, chunk_size):
    """
    Worker function for multiprocessing.
    """
    return encode_chunk(chunk, model, batch_size, chunk_size)

In [ ]:
# Disable tokenizer parallelism
os.environ["TOKENIZERS_PARALLELISM"] = "false"

gc.collect()
torch.cuda.empty_cache()
logging.info(f"Initial VRAM: {torch.cuda.memory_allocated() / 1024**3:.2f} GB")

# Initialize model (single instance for GPU)
model = SentenceTransformer('all-MiniLM-L6-v2').to('cuda')
logging.info(f"Model running on: cuda ({torch.cuda.get_device_name(0)})")

In [ ]:
# Read input file in chunks
try:
    chunk_iter = pd.read_csv(input_file, 
                            chunksize=chunk_size, 
                            engine='c',  sep='\t', encoding='utf-8', 
                            low_memory=False)
                            # on_bad_lines='skip')
    chunks = list(chunk_iter)  # Load chunks into memory (feasible with 512GB RAM)
    # logging.info(f"Loaded {len(chunks)} chunks")
except Exception as e:
    logging.error(f"Error reading CSV file: {e}")
    raise
# chunks = list(chunk_iter)  # Load chunks into memory (feasible with 512GB RAM)
logging.info(f"Loaded {len(chunks)} chunks")

In [ ]:
start = time.time()
# Process chunks in parallel
pool = mp.Pool(num_workers)
worker_func = partial(encode_chunk, model=model, batch_size=encode_batch_size, chunk_size=encode_chunk_size)

# Synchronize GPU access by processing chunks sequentially on the GPU
results = []
for chunk in tqdm(chunks, desc="Processing chunks"):
    result = worker_func(chunk)
    if result is not None:
        results.append(result)

pool.close()
pool.join()

# Combine results
if results:
    logging.info(f"Saving output to {output_file}")
    final_df = pd.concat(results, ignore_index=True)
    final_df['id'] = final_df['id'].astype('string')
    final_df.to_parquet(output_file, index=False, compression='snappy')
    logging.info(f"Output saved to {output_file}")

end = time.time()
logging.info(f"Total processing time: {end - start:.2f} seconds")


In [ ]:
"""
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity

# Assuming your DataFrame is named df and the two columns are 'column1' and 'column2'
tfidf = TfidfVectorizer().fit_transform(assigneeinstitution[['disambig_assignee_organization', 'affiliationame']].values.ravel())
pairwise_similarity = cosine_similarity(tfidf)
pairwise_similarity
"""

In [ ]:
institutionencodings = pd.read_parquet(dataint + "embeddedinstitutions.parquet")
institutionencodings['id'] =institutionencodings['id'].astype(str)
institutionencodings = institutionencodings.rename(columns={'encoded_title':'encoded_affiliationame', 'encoded_abstract':'encoded_disambig_assignee_organization'})
institutionencodings



In [ ]:
assigneeinstitutionencoding = pd.merge(assigneeinstitution, institutionencodings, on='id', how='inner')

assigneeinstitutionencoding

GPU-code the similarity using the encodings

In [ ]:
print(assigneeinstitutionencoding.shape)
assigneeinstitutionencoding = assigneeinstitutionencoding.dropna()
print(assigneeinstitutionencoding.shape)


In [ ]:
shapes = assigneeinstitutionencoding['encoded_disambig_assignee_organization'].apply(lambda x: np.array(x).shape if x is not None else None)
print(shapes.value_counts(dropna=False))

In [ ]:
import pandas as pd
import numpy as np
import torch
import torch.nn.functional as F

# Set device
device = 'cuda' if torch.cuda.is_available() else 'cpu'
print(f"Using device: {device}")

# Configurable batch size (adjust based on VRAM and data size)
BATCH_SIZE = 1000000  # ~1.23GB for 4 tensors with D=768, adjust as needed

# Assume pppapppapersembed_window is the input DataFrame
# Get total number of samples
num_samples = len(assigneeinstitutionencoding)
print(f"Total samples: {num_samples}")

# Initialize columns for results
assigneeinstitutionencoding['institution_similarity'] = np.nan

# Process data in batches
for start_idx in range(0, num_samples, BATCH_SIZE):
    end_idx = min(start_idx + BATCH_SIZE, num_samples)
    print(f"Processing batch: {start_idx} to {end_idx}")
    
    # Extract batch data
    batch_indices = range(start_idx, end_idx)
    batch_patent_assignee = torch.tensor(
        np.stack(assigneeinstitutionencoding['encoded_disambig_assignee_organization'].iloc[batch_indices].values),
        dtype=torch.float32
    ).to(device)
    batch_paper_institution = torch.tensor(
        np.stack(assigneeinstitutionencoding['encoded_affiliationame'].iloc[batch_indices].values),
        dtype=torch.float32
    ).to(device)
    
    # Compute cosine similarities
    institution_similarities = F.cosine_similarity(batch_patent_assignee, batch_paper_institution, dim=1)
    
    # Compute norms and handle zero-norm cases
    institution_norms = torch.norm(batch_patent_assignee, dim=1) * torch.norm(batch_paper_institution, dim=1)
    institution_similarities = torch.where(institution_norms == 0, torch.tensor(float('nan'), device=device), institution_similarities)
    
    # Store results in DataFrame
    assigneeinstitutionencoding.iloc[batch_indices, assigneeinstitutionencoding.columns.get_loc('institution_similarity')] = (
        institution_similarities.cpu().numpy()
    )
    
    # Clear GPU memory
    del batch_patent_assignee, batch_paper_institution
    del institution_similarities, institution_norms
    torch.cuda.empty_cache()

print("\nFinal DataFrame:")
print(assigneeinstitutionencoding[['id', 'affiliationame', 'disambig_assignee_organization', 'institution_similarity']])

In [ ]:
assigneeinstitutionencoding

In [ ]:
possiblepairsassigneesinstitutions

In [ ]:
possiblepairsassigneesinstitutions = pd.merge(possiblepairsassigneesinstitutions, assigneeinstitutionencoding[['disambig_assignee_organization', 'affiliationame', 'institution_similarity']], on=['disambig_assignee_organization', 'affiliationame'], how='left')
possiblepairsassigneesinstitutions

In [ ]:
print(possiblepairsassigneesinstitutions['affiliationame'].value_counts())
possiblepairsassigneesinstitutions.loc[
    possiblepairsassigneesinstitutions['affiliationame'] == 'nan', 'institution_similarity'
] = np.nan
possiblepairsassigneesinstitutions['institution_similarity'].isna().sum()


In [ ]:
possiblepairsassigneesinstitutions['institution_similarity'] = possiblepairsassigneesinstitutions['institution_similarity'].fillna(0)
possiblepairsassigneesinstitutions['institution_similarity'].isna().sum()

possiblepairsassigneesinstitutions.to_parquet(dataint + 'TEMPpossiblepairsassigneesinstitutionssimilarity.parquet', compression='snappy')


In [ ]:
assigneeinstitution

2-stage matching where quick similarity finds high-confidence matches and screens out low-confidence, then LLMs sort through the messy middle

In [ ]:
if usellm:
    import pandas as pd 
    dataint = './data/int/' 
    df_try=pd.read_parquet(dataint+'assigneeinstitution.parquet')

In [ ]:
# df_try=df_try[:1000]

In [ ]:
"""
import pandas as pd
import numpy as np
import ollama
from rapidfuzz.process import cdist
from concurrent.futures import ThreadPoolExecutor
import time
import logging
import torch
import gc
from tqdm import tqdm

# Setup logging
logging.basicConfig(filename='processing.log', level=logging.INFO, format='%(asctime)s - %(message)s')

# Configuration
ollamamodel = "llama3.1:8b-q4_0"  # Quantized model
batch_size = 128  # Optimized for RTX 4090
chunk_size = 100000
similarity_threshold = 90
junk_threshold = 40
dataraw = './data/raw/'
prompt_template = "Determine whether '{}' and '{}' are the same institution. Respond with 'Classification: Same' or 'Classification: Different'."

# Memory cleanup
def clear_memory():
    if torch.cuda.is_available():
        torch.cuda.empty_cache()
        gc.collect()

# Vectorized fuzzy matching
def vectorized_fuzzy_match(affiliations, orgs):
    affiliations = affiliations.str.lower().fillna('')
    orgs = orgs.str.lower().fillna('')
    # Compute similarity scores for corresponding pairs
    scores = np.array([fuzz.ratio(aff, org) for aff, org in zip(affiliations, orgs)])
    return np.where(scores >= similarity_threshold, 'Same',
                   np.where(scores < junk_threshold, 'Different', None))

# Batch LLM processing
def process_batch_prompt(batch_df, model=ollamamodel):
    try:
        prompts = [prompt_template.format(str(row['affiliationame']), str(row['disambig_assignee_organization'])) for _, row in batch_df.iterrows()]
        responses = ollama.generate(model=model, prompt=prompts, options={"num_gpu": -1, "num_ctx": 512, "temperature": 0.0})
        results = []
        for idx, (response, row) in enumerate(zip(responses, batch_df.itertuples())):
            response_text = response['response'].replace('\n', '')
            classification = 'Not mentioned' if not (row.affiliationame and row.disambig_assignee_organization) else \
                             response_text.split(':', 1)[1].strip() if 'classification:' in response_text.lower() else 'Not mentioned'
            results.append({
                "id": row.id,
                "affiliation": str(row.affiliationame),
                "assignee": str(row.disambig_assignee_organization),
                "response": response_text,
                "classification": classification
            })
        return results
    except Exception as e:
        logging.error(f"Error processing batch: {str(e)}")
        return []

# Main processing
start_time = time.time()
extracted_rows = []
df_try = df_try.astype({'affiliationame': 'string', 'disambig_assignee_organization': 'string', 'id': 'int32'})
df_try = df_try.fillna({'affiliationame': '', 'disambig_assignee_organization': ''})

for chunk_idx, start_idx in tqdm(enumerate(range(0, len(df_try), chunk_size)), total=len(df_try)//chunk_size + 1):
    chunk_df = df_try.iloc[start_idx:start_idx + chunk_size].copy()
    
    # Fuzzy matching
    chunk_df['classification'] = vectorized_fuzzy_match(chunk_df['affiliationame'], chunk_df['disambig_assignee_organization'])
    valid_mask = (chunk_df['affiliationame'].str.len() > 0) & (chunk_df['disambig_assignee_organization'].str.len() > 0)
    chunk_df.loc[~valid_mask, 'classification'] = 'Not mentioned'
    fuzzy_matched = chunk_df[chunk_df['classification'].notna()]
    to_process = chunk_df[chunk_df['classification'].isna() & valid_mask]
    
    logging.info(f"Chunk {chunk_idx + 1}: {len(fuzzy_matched)} rows fuzzy matched, {len(to_process)} rows for LLM")
    print(f"Chunk {chunk_idx + 1}: {len(fuzzy_matched)} rows fuzzy matched, {len(to_process)} rows for LLM")
    
    # LLM processing
    chunk_results = []
    if not to_process.empty:
        for batch_start in range(0, len(to_process), batch_size):
            batch_df = to_process.iloc[batch_start:batch_start + batch_size]
            chunk_results.extend(process_batch_prompt(batch_df, ollamamodel))
            logging.info(f"Chunk {chunk_idx + 1}, batch {batch_start // batch_size + 1}: Processed {len(batch_df)} rows")
            print(f"Processed {min(start_idx + batch_start + batch_size, len(df_try))} of {len(df_try)} rows")
    
    # Combine and save results
    fuzzy_results = [
        {"id": row['id'], "affiliation": row['affiliationame'], "assignee": row['disambig_assignee_organization'],
         "response": f"Fuzzy matched: {row['classification']}", "classification": row['classification']}
        for _, row in fuzzy_matched.iterrows()
    ]
    extracted_rows.extend(fuzzy_results + chunk_results)
    pd.DataFrame(fuzzy_results + chunk_results).to_parquet(f"{dataint}chunk_{chunk_idx + 1}_results.parquet", index=False, compression='snappy')
    
    clear_memory()

# Final output
result_df = pd.DataFrame(extracted_rows)
result_df.to_parquet(f"{dataint}llamafuzzsameinstitution.parquet", index=False, compression='snappy')
result_df.to_csv(f"{dataint}llamafuzzsameinstitution.tsv", index=False, sep='\t')
logging.info(f"Total processing time: {time.time() - start_time:.2f} seconds")
print(f"Total processing time: {time.time() - start_time:.2f} seconds")
"""

In [ ]:
if usellm:
    import pandas as pd
    import numpy as np
    import ollama
    from rapidfuzz import fuzz
    from concurrent.futures import ThreadPoolExecutor
    import time
    import logging
    import os

    logging.basicConfig(filename='processing.log', level=logging.INFO, format='%(asctime)s - %(message)s')

    ollamamodel = "llama3.1:8b"  # Updated to LLaMA 3.1 8B
    batch_size = 2048  # Increased for RTX 4090 (24GB VRAM)
    chunk_size = 500000  # Process 500k rows per chunk for 17M rows
    similarity_threshold = 90  # Fuzzy matching threshold for identical institutions
    junk_threshold = 40
    dataraw = './data/raw/'  # From prior context
    prompt_template = "Determine whether '{}' and '{}' are the same institution. Respond with 'Classification: Same' or 'Classification: Different' Do not explain your reasoning."

    # df_try = df_try.fillna({'affiliationame': '', 'disambig_assignee_organization': ''})
    def fuzzy_match(row):
        aff = str(row['affiliationame']).lower()
        org = str(row['disambig_assignee_organization']).lower()
        if not aff or not org:
            return 'Not mentioned'
        score = fuzz.ratio(aff, org)  # Case-insensitive similarity
        return 'Same' if score >= similarity_threshold else 'Different' if score < junk_threshold else None

    def process_prompt(row, model=ollamamodel):
        try:
            aff = str(row['affiliationame'])
            org = str(row['disambig_assignee_organization'])
            if not aff or not org:
                return None
            response = ollama.generate(model=model, prompt=prompt_template.format(aff, org), options={"num_gpu": -1, "num_ctx":512, "temperature": 0.0})
            response_text = response['response'].replace('\n', '')
            classification = 'Not mentioned'
            if 'classification:' in response_text.lower():
                classification = response_text.split(':', 1)[1].strip()
            return {
                "id": row['id'],
                "affiliation": aff,
                "assignee": org,
                "response": response_text,
                "classification": classification
            }
        except Exception as e:
            logging.error(f"Error processing row {row['id']}: {str(e)}")
            return None

    def process_batch(batch_df, model=ollamamodel):
        results = []
        with ThreadPoolExecutor(max_workers=16) as executor:
            batch_results = list(executor.map(process_prompt, [row for _, row in batch_df.iterrows()]))
        results.extend([r for r in batch_results if r is not None])
        return results

    start_time = time.time()
    extracted_rows = []

    for chunk_idx, start_idx in enumerate(range(0, len(df_try), chunk_size)):
        chunk_df = df_try.iloc[start_idx:start_idx + chunk_size].copy()
        chunk_df['classification'] = chunk_df.apply(fuzzy_match, axis=1)
        fuzzy_matched = chunk_df[chunk_df['classification'].notna()]
        to_process = chunk_df[chunk_df['classification'].isna()]
        logging.info(f"Chunk {chunk_idx + 1}: {len(fuzzy_matched)} rows matched by fuzzy, {len(to_process)} rows for LLM")
        print(f"Chunk {chunk_idx + 1}: {len(fuzzy_matched)} rows fuzzy matched, {len(to_process)} rows for LLM")
        chunk_results = []
        if not to_process.empty:
            for batch_start in range(0, len(to_process), batch_size):
                batch_df = to_process.iloc[batch_start:batch_start + batch_size]
                batch_results = process_batch(batch_df, ollamamodel)
                chunk_results.extend(batch_results)
                logging.info(f"Chunk {chunk_idx + 1}, batch {batch_start // batch_size + 1}: Processed {min(batch_start + batch_size, len(to_process))} rows")
                print(f"Processed {min(start_idx + batch_start + batch_size, len(df_try))} of {len(df_try)} rows")
        fuzzy_results = [
            {
                "id": row['id'],
                "affiliation": row['affiliationame'],
                "assignee": row['disambig_assignee_organization'],
                "response": f"Fuzzy matched: {row['classification']}",
                "classification": row['classification']
            } for _, row in fuzzy_matched.iterrows()
        ]
        extracted_rows.extend(fuzzy_results + chunk_results)
        pd.DataFrame(extracted_rows[-len(fuzzy_results + chunk_results):]).to_csv(
            f"{dataint}chunk_{chunk_idx + 1}_results.csv", index=False
        )
    result_df = pd.DataFrame(extracted_rows)
    result_df.to_parquet(f"{dataint}llamafuzzsameinstitution.parquet", index=False,compression='snappy')
    # result_df.to_csv(f"{dataint}llamafuzzsameinstitution.tsv", index=False, sep='\t')
    logging.info(f"Total processing time: {time.time() - start_time:.2f} seconds")
    print(f"Total processing time: {time.time() - start_time:.2f} seconds")
    import torch
    if torch.cuda.is_available():
        torch.cuda.empty_cache()
        torch.cuda.synchronize()

In [ ]:
if usellm:
    sameinstitutionflag = pd.read_parquet(dataint + 'llamafuzzsameinstitution.parquet')
    sameinstitutionflag 


In [ ]:
if usellm:
    sameinstitutionflag['sameinstitutionflag'] = sameinstitutionflag['classification'].apply(lambda x: 1 if x == 'Same' else 0)
    sameinstitutionflag = sameinstitutionflag.rename(columns={'assignee':'disambig_assignee_organization', 'affiliation':'affiliationame'})
    sameinstitutionflag

In [ ]:
possiblepairsassigneesinstitutions = pd.read_parquet(dataint + 'TEMPpossiblepairsassigneesinstitutions.parquet')

In [ ]:
if usellm:
    # assigneeinstitutionencoding = pd.merge(assigneeinstitution, institutionencodings, on='id', how='inner')

    possiblepairsassigneesinstitutions = pd.merge(possiblepairsassigneesinstitutions, sameinstitutionflag[['disambig_assignee_organization', 'affiliationame', 'sameinstitutionflag']], on=['disambig_assignee_organization', 'affiliationame'], how='left')

    possiblepairsassigneesinstitutions['sameinstitutionflag'] = possiblepairsassigneesinstitutions['sameinstitutionflag'].fillna(0)
    possiblepairsassigneesinstitutions['sameinstitutionflag'] = possiblepairsassigneesinstitutions['sameinstitutionflag'].astype(int)

    possiblepairsassigneesinstitutions


In [ ]:
# possiblepairsassigneesinstitutions = pd.read_parquet(dataint + 'possiblepairs_institutionoverlap.parquet')
# possiblepairsassigneesinstitutions.shape
possiblepairsassigneesinstitutions = pd.read_parquet(dataint + 'TEMPpossiblepairsassigneesinstitutionssimilarity.parquet')



In [ ]:
if usellm:
    possiblepairsassigneesinstitutions = possiblepairsassigneesinstitutions[['magid', 'patent_id', 'institution_similarity', 'sameinstitutionflag']]
    possiblepairsassigneesinstitutions['institution_similarity'] = possiblepairsassigneesinstitutions.groupby(['magid', 'patent_id'])['institution_similarity'].transform('max')
    possiblepairsassigneesinstitutions['sameinstitutionflag'] = possiblepairsassigneesinstitutions.groupby(['magid', 'patent_id'])['sameinstitutionflag'].transform('max')
    possiblepairsassigneesinstitutions = possiblepairsassigneesinstitutions.drop_duplicates()
    possiblepairsassigneesinstitutions = possiblepairsassigneesinstitutions.dropna()
    possiblepairsassigneesinstitutions.shape

In [ ]:
        # make sure we have a value for every pair
MISSINGinstitution_similarity = 0
if usellm:
    MISSINGsameinstitutionflag = 0
possiblepairsassigneesinstitutions2export = pd.merge(possiblepairsassigneesinstitutions, possiblepairs[['magid', 'patent_id']], on=['magid', 'patent_id'], how='right')
possiblepairsassigneesinstitutions2export['institution_similarity'] = possiblepairsassigneesinstitutions2export['institution_similarity'].fillna(MISSINGinstitution_similarity)
if usellm:
    possiblepairsassigneesinstitutions2export['sameinstitutionflag'] = possiblepairsassigneesinstitutions2export['sameinstitutionflag'].fillna(MISSINGsameinstitutionflag)
possiblepairsassigneesinstitutions2export

In [ ]:
possiblepairsassigneesinstitutions2export['institution_similarity_unadjusted']= possiblepairsassigneesinstitutions2export['institution_similarity']
if usellm:  
    possiblepairsassigneesinstitutions2export.loc[possiblepairsassigneesinstitutions2export['sameinstitutionflag'] == 1, 'institution_similarity'] = .99 
# possiblepairsassigneesinstitutions2export.loc[possiblepairsassigneesinstitutions2export['sameinstitutionflag'] == 1, 'institution_similarity'] = .95


In [ ]:
possiblepairsassigneesinstitutions2export.to_parquet(dataint + 'possiblepairs_institutionoverlap.parquet', index=False, compression='snappy')